# BSRNN workbook
The aim for this workbook is to start designing the components of the BSRNN from the 'High Fidelity Speech Enhancement BSRNN' paper.

In [29]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torchaudio
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
import soundfile as sf
import time
from collections import defaultdict
import math

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [ ]:
class TrialDataset(torch.utils.data.Dataset):
    def __init__(self, manifest_csv, data_root, split, chunk_s, sample_rate, seed, random_crop=True):
        self.data_root    = Path(data_root)
        self.split        = split                   
        self.chunk_s      = chunk_s        
        self.sample_rate  = sample_rate
        self.seed         = seed
        self.random_crop  = random_crop
        self.chunk_frames = int(chunk_s * sample_rate)
        self.epoch        = 0                       

        self.manifest_df = pd.read_csv(manifest_csv)

    def __len__(self):
        return len(self.manifest_df)

    def __getitem__(self, idx):
        # there are a couple of things that I need to get:
        # 1. the mixture audio
        # 2. the target speaker audio
        # 3. the enrollment audio

        # Then there are a few things that are not necessary but are useful:
        # 1. the meta data
        # 2. the trial_id
        # 3. the crop absent

        row = self.manifest_df.iloc[idx]
        trial_directory = self.data_root / "rendered" / self.split / row["trial_id"]
        mixture_directory = trial_directory / "mixture.wav"
        number_of_frames = sf.info(str(mixture_directory)).frames

        start_offset = self._crop_offset_start(idx, number_of_frames)
        mixture_audio = self._read_in_wav(mixture_directory, start=start_offset, frames=self.chunk_frames)
        target_audio = self._read_in_wav(trial_directory / "target.wav", start=start_offset, frames=self.chunk_frames)
        enrollment_audio = self._read_in_wav(trial_directory / "enrollment.wav")

        crop_absent = bool(target_audio.abs().max() == 0)
        return {
            "mixture": mixture_audio,
            "target": target_audio,
            "enrollment": enrollment_audio,
            "crop_absent": crop_absent,
            "trial_id": str(row["trial_id"]),
            "meta": {
                "condition":        str(row["condition"]),
                "clip_absent":      bool(row["target_absent"]),
                "sir_db":           float(row["sir_db"]),
                "snr_db":           float(row["snr_db"]),
                "overlap_achieved": float(row["overlap_achieved"]),
                "regime":           str(row["regime"]),
                "same_gender":      float(row["same_gender"]),
            },
        }


    def _read_in_wav(self, path, start=0, frames=-1):
        # Read in a wav file and return a torch tensor of shape (frames,)
        with sf.SoundFile(str(path)) as f:
            assert f.samplerate == self.sample_rate, f"Sample rate mismatch: {f.samplerate} != {self.sample_rate}"
            assert f.channels == 1, f"Channel mismatch: {f.channels} != 1"

            if start > 0:
                f.seek(start)
            
            x = f.read(frames, dtype='float32', always_2d=False)

        return torch.from_numpy(np.ascontiguousarray(x))

    def _crop_offset_start(self, idx, n_frames):
        # This is used to determine the starting point of the crop for the audio. It is important to note that this is only used for the mixture and target audio, not the enrollment audio. The enrollment audio is always read in full.

        max_start = n_frames - self.chunk_frames
        assert max_start >= 0, f"clip {n_frames} shorter than chunk {self.chunk_frames}"

        epoch = self.epoch if self.random_crop else 0
        rng = np.random.default_rng((self.seed, epoch, idx))
        return int(rng.integers(0, max_start + 1))

    def set_epoch(self, epoch):
        # call this at the top of each training epoch to ensure that the random cropping is consistent across all samples in the dataset. This is important for reproducibility and to ensure that the model sees the same data in each epoch.
        self.epoch = epoch


In [ ]:
smoke_train_dataset = TrialDataset(
    manifest_csv="./../../data/manifests/smoke_train.csv",
    data_root="./../../data",
    split="smoke_train",
    chunk_s=4.0, # follow CARTSE
    sample_rate=16000,
    seed=42
)

smoke_valid_dataset = TrialDataset(
    manifest_csv="./../../data/manifests/smoke_val.csv",
    data_root="./../../data",
    split="smoke_val",
    chunk_s=4.0, # follow CARTSE
    sample_rate=16000,
    seed=42,
    random_crop=False,
)

smoke_train_loader = torch.utils.data.DataLoader(
    smoke_train_dataset,
    batch_size=12, # Follow CARTSE
    shuffle=True,
    num_workers=4
)

smoke_valid_loader = torch.utils.data.DataLoader(
    smoke_valid_dataset,
    batch_size=12, # Follow CARTSE
    shuffle=False,
    num_workers=4
)

In [22]:
# Example of how to use the dataset and dataloader
for batch in smoke_train_loader:
    mixture = batch["mixture"]
    target = batch["target"]
    enrollment = batch["enrollment"]
    crop_absent = batch["crop_absent"]
    trial_id = batch["trial_id"]
    meta = batch["meta"]

    print(f"Mixture shape: {mixture.shape}")
    print(f"Target shape: {target.shape}")
    print(f"Enrollment shape: {enrollment.shape}")
    print(f"Crop absent: {crop_absent}")
    print(f"Trial ID: {trial_id}")
    print(f"Meta: {meta}")
    break  # Just process one batch for demonstration 

Mixture shape: torch.Size([12, 64000])
Target shape: torch.Size([12, 64000])
Enrollment shape: torch.Size([12, 80000])
Crop absent: tensor([False, False,  True, False, False, False,  True,  True, False, False,
        False, False])
Trial ID: ['smoke_train-42-000011', 'smoke_train-42-000049', 'smoke_train-42-000003', 'smoke_train-42-000030', 'smoke_train-42-000026', 'smoke_train-42-000004', 'smoke_train-42-000040', 'smoke_train-42-000009', 'smoke_train-42-000048', 'smoke_train-42-000008', 'smoke_train-42-000020', 'smoke_train-42-000042']
Meta: {'condition': ['both', 'target_only', 'noise_only', 'both', 'target_only', 'both', 'interferer_only', 'interferer_only', 'both', 'target_only', 'both', 'both'], 'clip_absent': tensor([False, False,  True, False, False, False,  True,  True, False, False,
        False, False]), 'sir_db': tensor([7.3200,    nan,    nan, 3.6400,    nan, 1.2100,    nan,    nan, 8.0300,
           nan, 3.4400, 8.9600], dtype=torch.float64), 'snr_db': tensor([19.2700, 

In [24]:
# Need to just validate that the crops that I make still align to the split of data with the 50% both, 25% target only, 20% interfere only, and 5% noise only
def measure_empty_crops(ds, n_trials=3000, epochs=3, sample_seed=0):
    """Fraction of crops that contain no target speech, per epoch and condition.
    
    Reuses the dataset's own crop-offset RNG so the offsets are exactly the ones
    training will draw. Reads only target.wav — mixture and enrollment are not
    needed to answer this.
    """
    rng = np.random.default_rng(sample_seed)
    idxs = rng.choice(len(ds), size=min(n_trials, len(ds)), replace=False)
    
    n_frames_cache = {}          # frames don't change across epochs
    rows = []
    t0 = time.perf_counter()
    
    for ep in range(epochs):
        ds.set_epoch(ep)
        for i in idxs:
            i = int(i)
            r = ds.manifest_df.iloc[i]
            tdir = ds.data_root / "rendered" / ds.split / r["trial_id"]
            
            if i not in n_frames_cache:
                n_frames_cache[i] = sf.info(str(tdir / "mixture.wav")).frames
            start = ds._crop_offset_start(i, n_frames_cache[i])
            
            tgt = ds._read_in_wav(tdir / "target.wav",
                                start=start, frames=ds.chunk_frames)
                                 
            rows.append({         
                "epoch":       ep,
                "condition":   r["condition"],
                "clip_absent": bool(r["target_absent"]),
                "crop_absent": bool(tgt.abs().max() == 0),
                "activity":    float(r["target_activity"]),
            })  
              
    print(f"{len(rows)} crops in {time.perf_counter()-t0:.1f}s")
    return pd.DataFrame(rows)                    


In [ ]:
# train_dataset = TrialDataset(
#     manifest_csv="./../../data/manifests/train.csv",
#     data_root="./../../data",
#     split="train",
#     chunk_s=4.0,
#     sample_rate=16000,
#     seed=42,
#     random_crop=True,
# )

# df = measure_empty_crops(train_dataset, n_trials=3000, epochs=3)

# designed  = df["clip_absent"].mean()      # what the generator was configured for
# effective = df["crop_absent"].mean()      # what the model actually trains on
# present   = df[~df["clip_absent"]]
# leakage   = present["crop_absent"].mean()

# print(f"designed  absent rate (clip level) : {designed:.3f}")
# print(f"effective absent rate (crop level) : {effective:.3f}")
# print(f"drift                              : {effective - designed:+.3f}")
# print(f"leakage (present trial -> silent crop): {leakage:.3f}")

# print("\nleakage by condition:")
# print(present.groupby("condition")["crop_absent"].agg(["mean", "count"]))

# print("\nleakage by target_activity band:")
# banded = present.assign(band=pd.cut(present["activity"], [0, .2, .3, .4, .5, .6, .8]))
# print(banded.groupby("band", observed=True)["crop_absent"].agg(["mean", "count"]))

# print("\nper-epoch effective rate (should be stable):")
# print(df.groupby("epoch")["crop_absent"].mean())


## 1. Create the STFT

In [ ]:
class STFT(nn.Module):
    def __init__(self, n_fft=512, hop_length=128, sample_rate=16000):
        super().__init__()
        self.n_fft = n_fft
        self.hop_length = hop_length
        self.sample_rate = sample_rate
        self.padding = n_fft - hop_length
        self.register_buffer("window", torch.hann_window(n_fft), persistent=False)
    
          
    def latency_ms(self, lookahead_frames=0):
        """Algorithmic latency. Convention: window fill + emit hop + lookahead.
        State the convention when you quote it -- CARTSE counts window - hop
        instead, so their 24 ms and our 40 ms describe the same framing."""
        samples = self.n_fft + self.hop_length + lookahead_frames * self.hop_length
        return 1000 * samples / self.sample_rate 

    def _n_frames(self, T):
      # +2*padding: the tail needs the same ramp-out room the head gets, or the
      # final samples land where the OLA envelope has decayed to ~0.
      return math.ceil((T + 2 * self.padding - self.n_fft) / self.hop_length) + 1
 
    # (B, T) -> (B, F, N) complex
    def forward(self, x):                        
        T = x.shape[-1]
        n_frames = self._n_frames(T)
        right = (n_frames - 1) * self.hop_length + self.n_fft - (T + self.padding)
        xp = F.pad(x, (self.padding, right))
        return torch.stft(xp, self.n_fft, self.hop_length, self.n_fft,
                            self.window, center=False, return_complex=True)

    # (B, F, N) complex -> (B, length)                  
    def inverse(self, X, length):              
        frames = torch.fft.irfft(X, n=self.n_fft, dim=1) * self.window[None, :, None]
        B, _, N = frames.shape
        total = (N - 1) * self.hop_length + self.n_fft
        fold = lambda t: F.fold(t, (1, total), (1, self.n_fft), stride=(1, self.hop_length))
        
        out = fold(frames).view(B, total)
        env = fold(self.window.pow(2)[None, :, None].expand(1, -1, N)).view(total)
        return (out / env.clamp_min(1e-11))[:, self.padding:self.padding + length]


# this will be used in the model!! ie will be used:
#   sep_out = self.separator(subband_feature)
#   sep_out = lookahead_shift(sep_out, self.lookahead_frames)
#   mask    = self.band_masker(sep_out, subband_mix_spec)

def lookahead_shift(h, k):
    """Give the mask head k frames of future context.
    
    h: (B, N, T) features out of the causal sequence stack.
    Returns h' where h'[..., t] == h[..., t + k], the last k frames edge-padded.

    The mask for frame t is then built from a hidden state that has consumed
    frames up to t + k, i.e. k * hop of lookahead -- while the mask itself stays
    aligned to mixture frame t, which is required because a multiplicative mask
    cannot shift energy in time.
    """
    if k == 0:
        return h
    return F.pad(h[..., k:], (0, k), mode="replicate")


In [33]:
# example test STFT
f = STFT()
x = torch.randn(2, 64000)
print("round-trip:", (f.inverse(f(x), 64000) - x).abs().max().item())   # expect ~7e-7
print("latency k=0:", f.latency_ms(0), "ms   k=16:", f.latency_ms(16), "ms")

round-trip: 9.5367431640625e-07
latency k=0: 40.0 ms   k=16: 168.0 ms


## 2. Build the band plan table as a pure function
